#### Based on usage optimize table; experience

In [0]:
%sql
-- Enable for a specific catalog
ALTER CATALOG main ENABLE PREDICTIVE OPTIMIZATION;

-- Enable for a specific schema
ALTER SCHEMA main.default ENABLE PREDICTIVE OPTIMIZATION;

-- Check if it's enabled for a specific table
DESCRIBE DETAIL main.default.my_table;


In [0]:
%sql
-- Create table with explicit clustering columns
CREATE TABLE main.default.sales (
  order_id INT,
  customer_id INT,
  order_date DATE
) CLUSTER BY (customer_id, order_date);

-- Create table with AI-driven automatic clustering
CREATE TABLE main.default.web_logs (
  user_id STRING,
  event_type STRING,
  timestamp TIMESTAMP
) CLUSTER BY AUTO; -- Requires Predictive Optimization


In [0]:
%sql
-- Add clustering to an existing table
ALTER TABLE main.default.sales CLUSTER BY (order_date, customer_id);

-- Change clustering keys (only affects future data/optimizations)
ALTER TABLE main.default.sales CLUSTER BY (customer_id);

-- Force immediate reclustering of all data
OPTIMIZE main.default.sales FULL; --


#### ZORDER BY

In [0]:
%sql
-- Manually cluster data by specific columns for better data skipping
OPTIMIZE main.default.legacy_table
ZORDER BY (category, transaction_date);


#### Partitioning

In [0]:
%sql
-- Broadcast hint to avoid large shuffles (good for small dimension tables)
SELECT /*+ BROADCAST(small_table) */ *
FROM large_table
JOIN small_table ON large_table.id = small_table.id;

-- Adjust shuffle partition count for small/medium workloads
SET spark.sql.shuffle.partitions = 200;


#### VACUUM

In [0]:
%sql
-- Extend retention to 30 days so VACUUM doesn't delete too aggressively
ALTER TABLE main.default.sales SET TBLPROPERTIES (
  'delta.deletedFileRetentionDuration' = 'interval 30 days'
);


#### OPTIMIZE

In [0]:
%sql
OPTIMIZE events;

OPTIMIZE events FULL;

OPTIMIZE events WHERE date >= '2017-01-01';

OPTIMIZE events
    WHERE date >= current_timestamp() - INTERVAL 1 day
    ZORDER BY (eventType);